In [1]:
import pandas as pd
import pandas_gbq
import numpy as np
import itertools
import datetime
import matplotlib.pyplot as plt
from datetime import datetime
from datetime import *
from datetime import datetime, timedelta, date
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)
pd.set_option('display.max_colwidth', None)  # Show full column width


# Standard plotly imports
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)

In [2]:
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [3]:
#Change the date to the previous day
query = """
WITH Player_data as (
SELECT 
Season,
GAME_ID,
PLAYER_NAME,
PLAYER_ID,
TEAM_NAME,
MIN,
FROM `nba_api_player_boxscore.player_box_96_onwards`
ORDER BY 4
),

Draft_data as 
(
SELECT 
PLAYER_NAME, PERSON_ID as PLAYER_ID, SEASON, OVERALL_PICK, TEAM_NAME
FROM 
`perceptive-ivy-290216.nba_api_draft.draft_history`
ORDER BY Season DESC, OVERALL_PICK
)

SELECT
A.*, B.OVERALL_PICK, B.SEASON AS Draft_Year
FROM
Player_data A
LEFT JOIN
Draft_data B
ON A.PLAYER_NAME=B.PLAYER_NAME AND A.PLAYER_ID=B.PLAYER_ID
WHERE A.SEASON>='2001-02'
ORDER BY Draft_Year
"""
project_id = "perceptive-ivy-290216"
df_bq = pandas_gbq.read_gbq(query, project_id=project_id, dialect='standard')

Downloading: 100%|██████████|


In [14]:
game=df_bq

In [15]:
game.tail(10)

,Season,GAME_ID,PLAYER_NAME,PLAYER_ID,TEAM_NAME,MIN,OVERALL_PICK,Draft_Year
603486,2025-26,0022500045,Jeremiah Fears,1642847,New Orleans Pelicans,29,7,2025
603487,2025-26,0022500045,Derik Queen,1642852,New Orleans Pelicans,29,13,2025
603488,2025-26,0022500045,Micah Peavy,1642877,New Orleans Pelicans,7,40,2025
603489,2025-26,0022500237,Jeremiah Fears,1642847,New Orleans Pelicans,25,7,2025
603490,2025-26,0022500237,Derik Queen,1642852,New Orleans Pelicans,25,13,2025
603491,2025-26,0022500237,Micah Peavy,1642877,New Orleans Pelicans,4,40,2025
603492,2025-26,0022500044,Joan Beringer,1642866,Minnesota Timberwolves,1,17,2025
603493,2025-26,0022500025,Ace Bailey,1642846,Utah Jazz,23,5,2025
603494,2025-26,0022500098,Walter Clayton Jr.,1642383,Utah Jazz,20,18,2025
603495,2025-26,0022500098,Ace Bailey,1642846,Utah Jazz,13,5,2025


In [ ]:
#Get total and mean minutes played by pick number
game_agg = game.groupby(['OVERALL_PICK']).agg(
    min_mean=('MIN', 'mean'), 
    min_sum=('MIN', 'sum')   
).reset_index() # Reset index to turn back into columns

game_agg.head()

,OVERALL_PICK,min_mean,min_sum
0,1,31.70171,591332
1,2,28.575708,492188
2,3,30.901215,569633
3,4,28.817613,534365
4,5,27.473007,514487


In [45]:
fig_game_mean = px.bar(
  game_agg[game_agg["OVERALL_PICK"]<61], 
  x="OVERALL_PICK", 
  y=["min_mean"],
  # color="PLAYER_NAME_LAST_FIRST",
  template='plotly_white',
  title="<b>Average Minutes Per Game by Draft Pick</b>", 
#   line_shape='spline',
  height=500, 
  width=1400,
  )
fig_game_mean.update_layout(
    # showlegend=False,
    title_x=0.5,
    xaxis_title="Pick Number",
    yaxis_title="Average Minutes Per Game",
    hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=15)),
    xaxis = dict(tickfont = dict(size=15)),
    font=dict(
        family="PT Sans Narrow",
        size=14,
        color="Black"
    ),
    title_font_family="PT Sans Narrow"
)

fig_game_mean.update_traces(marker_color='#46C1F0')
fig_game_mean

In [46]:
fig_game_sum = px.bar(
  game_agg[game_agg["OVERALL_PICK"]<61], 
  x="OVERALL_PICK", 
  y=["min_sum"],
  # color="PLAYER_NAME_LAST_FIRST",
  template='plotly_white',
  title="<b>Total Minutes by Draft Pick</b>", 
#   line_shape='spline',
  height=500, 
  width=1400,
  )
fig_game_sum.update_layout(
    # showlegend=False,
    title_x=0.5,
    xaxis_title="Pick Number",
    yaxis_title="Total Minutes",
    hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=15)),
    xaxis = dict(tickfont = dict(size=15)),
    font=dict(
        family="PT Sans Narrow",
        size=14,
        color="Black"
    ),
    title_font_family="PT Sans Narrow"
)

fig_game_sum.update_traces(marker_color='#46C1F0')

fig_game_sum

In [50]:
#Get season and game counts by player and draft pick
season_game_count_by_player = game.groupby(['PLAYER_NAME','OVERALL_PICK']).agg(
    season_cnt=('Season', 'nunique'), 
    game_cnt=('GAME_ID', 'nunique')   
).reset_index() # Reset index to turn back into columns

#Aggregate Season and game count by draft pick
season_game_count_by_pick = season_game_count_by_player.groupby(['OVERALL_PICK']).agg(
    seasons_avg=('season_cnt', 'mean'), 
    games_avg=('game_cnt', 'mean'),
    games_total=('game_cnt', 'sum')  
).reset_index() # Reset index to turn back into columns
season_game_count_by_pick.head()

,OVERALL_PICK,seasons_avg,games_avg,games_total
0,1,8.487179,478.282051,18653
1,2,8.000000,465.513514,17224
2,3,8.828571,526.685714,18434
3,4,8.638889,515.083333,18543
4,5,8.210526,492.815789,18727


In [48]:
fig_pick_season_cnt = px.bar(
  season_game_count_by_pick[season_game_count_by_pick["OVERALL_PICK"]<61], 
  x="OVERALL_PICK", 
  y=["seasons_avg"],
  # color="PLAYER_NAME_LAST_FIRST",
  template='plotly_white',
  title="<b>Average Number of Seasons played by Draft Pick Position</b>", 
#   line_shape='spline',
  height=500, 
  width=1400,
  )
fig_pick_season_cnt.update_layout(
    # showlegend=False,
    title_x=0.5,
    xaxis_title="Pick Number",
    yaxis_title="Average Seasons Played",
    hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=15)),
    xaxis = dict(tickfont = dict(size=15)),
    font=dict(
        family="PT Sans Narrow",
        size=14,
        color="Black"
    ),
    title_font_family="PT Sans Narrow"
)

fig_pick_season_cnt.update_traces(marker_color='#46C1F0')

fig_pick_season_cnt

In [52]:
fig_pick_games_cnt = px.bar(
  season_game_count_by_pick[season_game_count_by_pick["OVERALL_PICK"]<61], 
  x="OVERALL_PICK", 
  y=["games_avg"],
  # color="PLAYER_NAME_LAST_FIRST",
  template='plotly_white',
  title="<b>Average Number of Games played by Draft Pick Position</b>", 
#   line_shape='spline',
  height=500, 
  width=1400,
  )
fig_pick_games_cnt.update_layout(
    # showlegend=False,
    title_x=0.5,
    xaxis_title="Pick Number",
    yaxis_title="Average Games Played",
    hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=15)),
    xaxis = dict(tickfont = dict(size=15)),
    font=dict(
        family="PT Sans Narrow",
        size=14,
        color="Black"
    ),
    title_font_family="PT Sans Narrow"
)

fig_pick_games_cnt.update_traces(marker_color='#46C1F0')

fig_pick_games_cnt

In [ ]:
fig_pick_games_cnt = px.bar(
  season_game_count_by_pick[season_game_count_by_pick["OVERALL_PICK"]<61], 
  x="OVERALL_PICK", 
  y=["games_total"],
  # color="PLAYER_NAME_LAST_FIRST",
  template='plotly_white',
  title="<b>Total Number of Games played by Draft Pick Position</b>", 
#   line_shape='spline',
  height=500, 
  width=1400,
  )
fig_pick_games_cnt.update_layout(
    # showlegend=False,
    title_x=0.5,
    xaxis_title="Pick Number",
    yaxis_title="Total Games Played",
    hoverlabel=dict(
        bgcolor="white",
        font_size=16,
        font_family="PT Sans Narrow"
    ),
    yaxis = dict(tickfont = dict(size=15)),
    xaxis = dict(tickfont = dict(size=15)),
    font=dict(
        family="PT Sans Narrow",
        size=14,
        color="Black"
    ),
    title_font_family="PT Sans Narrow"
)

fig_pick_games_cnt.update_traces(marker_color='#46C1F0')

fig_pick_games_cnt

In [58]:
#Change the date to the previous day
query_starter_bench = """
WITH Player_data as (
SELECT 
TEAM,
GAME_ID,
PLAYER_NAME,
personId as PLAYER_ID,
SAFE_CAST(SPLIT(Season, '-')[SAFE_OFFSET(0)] as STRING) Season, 
position, 
CASE WHEN position IN ('G','C','F') THEN 'Starter' ELSE 'Bench' END as Starter,
minutes,
ROUND((SAFE_CAST(SPLIT(minutes, ':')[SAFE_OFFSET(0)] as INT64)*60 + SAFE_CAST(SPLIT(minutes, ':')[SAFE_OFFSET(1)] as INT64))/60,2) as Total_Mins,
FROM `perceptive-ivy-290216.nba_api_box_traditional.reg_season_25_26`
ORDER BY 4
),

Draft_data as 
(
SELECT 
PLAYER_NAME, PERSON_ID as PLAYER_ID, SEASON, OVERALL_PICK, TEAM_NAME
FROM 
`perceptive-ivy-290216.nba_api_draft.draft_history`
ORDER BY Season DESC, OVERALL_PICK
)

SELECT
A.*, B.OVERALL_PICK, B.SEASON AS Draft_Year
FROM
Player_data A
JOIN
Draft_data B
ON A.PLAYER_NAME=B.PLAYER_NAME AND A.PLAYER_ID=B.PLAYER_ID AND A.SEASON=B.SEASON
ORDER BY Draft_Year
"""
project_id = "perceptive-ivy-290216"
df_bq_starter_bench = pandas_gbq.read_gbq(query_starter_bench, project_id=project_id, dialect='standard')

Downloading: 100%|██████████|


In [59]:
df_bq_starter_bench.head()

,TEAM,GAME_ID,PLAYER_NAME,PLAYER_ID,Season,position,Starter,minutes,Total_Mins,OVERALL_PICK,Draft_Year
0,New Orleans Pelicans,0022500219,Derik Queen,1642852,2025,,Bench,37:45,37.75,13,2025
1,New Orleans Pelicans,0022500095,Derik Queen,1642852,2025,,Bench,34:45,34.75,13,2025
2,New Orleans Pelicans,0022500117,Derik Queen,1642852,2025,,Bench,21:41,21.68,13,2025
3,New Orleans Pelicans,0022500027,Derik Queen,1642852,2025,,Bench,12:39,12.65,13,2025
4,New Orleans Pelicans,0022500168,Derik Queen,1642852,2025,,Bench,17:58,17.97,13,2025


In [60]:
starter_bench_agg = df_bq_starter_bench.groupby(['OVERALL_PICK','Starter']).agg(
    game_cnt=('GAME_ID', 'nunique'), 
).reset_index() # Reset index to turn back into columns

starter_bench_agg.head()

,OVERALL_PICK,Starter,game_cnt
0,1,Starter,15
1,2,Bench,6
2,3,Starter,13
3,4,Bench,1
4,4,Starter,13


In [61]:
starter_bench_agg

,OVERALL_PICK,Starter,game_cnt
0,1,Starter,15
1,2,Bench,6
2,3,Starter,13
3,4,Bench,1
4,4,Starter,13
5,5,Bench,9
6,5,Starter,4
7,6,Bench,9
8,6,Starter,4
9,7,Bench,2
